# 03 EDA Planning

Notebook nay chot storytelling, chart plan, va join checklist cho phan business EDA.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'dataset').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root from the current working directory.')

PROJECT_ROOT = find_project_root(Path.cwd()).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.competition_workflow import build_eda_plan_tables, export_eda_plan_artifacts

pd.set_option('display.max_colwidth', 120)

In [2]:
plan_tables = build_eda_plan_tables()
for table_name, table_df in plan_tables.items():
    print(f'\n=== {table_name} ===')
    display(table_df)


=== theme_plan ===


,theme,business_question,tables_needed,primary_metrics,recommended_charts,expected_insight,recommendation
0,Theme 1 - Revenue & Demand,Revenue changes over time and the strongest product/category demand drivers.,"sales, orders, order_items, products","Monthly revenue, YoY growth, category share, AOV","Time-series line, seasonality heatmap, stacked category bars, net revenue ranking",Identify peak periods and the categories or segments carrying the topline.,Shift inventory and campaign budget toward peak windows and top categories.
1,Theme 2 - Customer & Channel,"Which channels, devices, and regions drive efficient conversion and repeat demand?","web_traffic, orders, customers, geography","Bounce rate, session-to-order rate, orders per customer, revenue by region","Bounce-rate bar, source scatter, device mix bar, revenue by region bar/map",Find low-quality traffic sources and the channel-region combinations worth scaling.,Reallocate spend toward efficient sources and reinforce high-performing regions.
2,Theme 3 - Returns & Customer Experience,Which products drive returns and how do logistics affect customer satisfaction?,"returns, order_items, products, reviews, shipments","Return rate, return reasons, average rating, refund amount, shipping delay","Return heatmap, return-reason bar, delay-rating scatter, rating box plot","Detect fit, quality, or fulfillment issues that increase churn risk.","Improve size guidance, tighten logistics SLAs, and prioritize problematic SKUs."
3,Theme 4 - Inventory & Operations,Where are stockouts or overstock reducing revenue efficiency?,"inventory, order_items, products, sales","Stockout days, fill rate, sell-through rate, lost revenue estimate","Stockout bar, fill-rate heatmap, sell-through ranking, lost-revenue bar",Quantify operational bottlenecks and the categories most exposed to stock risk.,Raise safety stock for constrained categories and unwind persistent overstock.



=== chart_plan ===


,theme,chart_name,chart_type,grain,output_file
0,Theme 1,Monthly revenue trend,line,month,charts/theme1_monthly_revenue.png
1,Theme 1,Revenue seasonality matrix,heatmap,month x year,charts/theme1_revenue_seasonality.png
2,Theme 1,Quarterly category contribution,stacked bar,quarter x category,charts/theme1_category_contribution.png
3,Theme 1,Top segments by net revenue,bar,segment,charts/theme1_segment_net_revenue.png
4,Theme 2,Bounce rate by traffic source,bar,traffic_source,charts/theme2_bounce_rate_by_source.png
5,Theme 2,Sessions vs orders by source,scatter,traffic_source,charts/theme2_sessions_vs_orders.png
6,Theme 2,Order count by device type,bar,device_type,charts/theme2_orders_by_device.png
7,Theme 2,Revenue by region,bar,region,charts/theme2_revenue_by_region.png
8,Theme 3,Return rate by size and category,heatmap,size x category,charts/theme3_return_rate_heatmap.png
9,Theme 3,Return reasons for Streetwear vs other segments,bar,return_reason,charts/theme3_return_reasons.png



=== join_preparation ===


,theme,join_instruction
0,Theme 1,"Join order_items to orders on order_id, then enrich with products on product_id."
1,Theme 1,Use sales as the authoritative daily target table for revenue and COGS.
2,Theme 2,Join orders to customers on customer_id and geography on zip.
3,Theme 2,Aggregate web_traffic by date and traffic_source before channel comparison.
4,Theme 3,Join returns to order_items and products on order_id + product_id to recover size and segment.
5,Theme 3,Join reviews and shipments on order_id to compute delay-rating relationships.
6,Theme 4,Use inventory snapshot_date monthly aggregates and enrich with products when needed.
7,Theme 4,Estimate lost revenue by combining inventory stockout exposure with realized category revenue.



=== insight_taxonomy ===


,insight,tier,priority
0,Revenue seasonality peaks in Q4,Descriptive,1
1,A traffic source has high sessions but weak conversion efficiency,Diagnostic,2
2,Specific sizes or segments carry outsized return rates,Diagnostic,2
3,Stockouts suppress potential category revenue,Diagnostic,3
4,Lower shipping delay is associated with higher customer ratings,Predictive,4
5,Budget reallocation toward efficient channels can improve ROI,Prescriptive,5


## Narrative reminders

- Moi insight nen di theo truc: What -> So what -> Implication -> Action.
- Uu tien diagnostic va prescriptive insight, khong dung lai o chart mo ta.
- Giu style chart nhat quan ve mau, format so, va tieu de.

In [3]:
artifact_paths = export_eda_plan_artifacts(PROJECT_ROOT)
pd.DataFrame({
    'artifact': list(artifact_paths.keys()),
    'path': [str(path.relative_to(PROJECT_ROOT)) for path in artifact_paths.values()],
}).sort_values('artifact')

,artifact,path
1,chart_plan,artifacts\eda_planning\chart_plan.csv
3,insight_taxonomy,artifacts\eda_planning\insight_taxonomy.csv
2,join_preparation,artifacts\eda_planning\join_preparation.csv
0,theme_plan,artifacts\eda_planning\theme_plan.csv
